# Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

 - Tracking agent behavior with logging, analytics, and debugging.
 - Transforming prompts, tool selection, and output formatting.
 - Adding retries, fallbacks, and early termination logic.
 - Applying rate limits, guardrails, and PII detection.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

 - Long-running conversations that exceed context windows.
 - Multi-turn dialogues with extensive history.
 - Applications where preserving full conversation context matters.

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


### Messagebased summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [5]:
### Run with thread
config = {"configurable": {"thread_id": "test_1"}}

In [6]:
### Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*0?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messaages: {response}")
    print(f"Messages: {len(response["messages"])}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messaages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a5326944-2177-419f-8fcb-bd7f0a4a2e22'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 is 4.', 'extras': {'signature': 'EoADCv0CAWkUfRMdvJ2Rtk8w3ZhUj4etQtO50QOt9oxFdCUNdNtNSwS9nwnlaaqw8tFJ3SRV8JWVZx3imupSzaJRB78ISvYcIk9yHt/x2k8dHFiMoCcmLuJOi7Euie0VZ0Ca7quaTIbg5lwnN0S5hph3yPtb+Z4JLHF7w1h/sKqJHtLvMg3lEh8TNPB/XCBfbshWWRGXwPda2cIDzhgypoikrCGIPfzsQwOBkfG17xSrLdoJsu+z/d1ZSl9/8HezbLqBrB/tHHb2JiSwXNv1jvX03miS6CAkfdBqNzKZCVq35xV73vhXePJpNiNLoHgy1LD3p++Gbjz9JdGwMFSlyFAjMB4pZwGvV27HKlR8nWSRkn5YlWNJowFTZT4RZXLKnzd0Rw3sDwk5Q80rbE4yh/PSEskE/+U2HDH2o2T4kliGwlSYI14aHyvxasYM2wyc2CRLEr9U8e1IPJQprsPlQsKYTi3GA245Zb7U1Mrngx/QxEtPLRuCJan1BhRlKRLtGC+s'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5be-9a8d-7fb0-b2b7-cc8321a1c829-0', tool_calls=[], invalid_tool_calls=

### Token size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

### Tokenbased summarization
agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test_1"}}

# Token Counter (approximate)
def count_tokens(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4   # 4 char = 1 token

In [9]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~159 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='5a20eeee-0612-4b94-a32e-8ad3761953e5'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_601613': 'EpoDCpcDAWkUfRPwjRFFlsIPpWibkoRLVr6DfW1SbXbUWvDnwkgajCQHtjP1kM4yOI+0NsEmjdk8xVBhnPuD26zLVo8c6FYRznOxN3SeMseiLhzJY9H6a7rB0nWke4xDi63rcqkogq1gFxIi0IDknH+WexL5l6AFE4hU3HMDzFdCOd5/noX61WyeTCXx8R3nrbpGxAc1dRQAZbm5VGwXBWB8/ZuM1QJlKkChKV8dsByUBseMawlQvNFSWOYMf3iOcwMDqG/Ig3VpSYDmiHTvKS8XIU0z1FQDMfcCgT8Z3OIIFMRPiIgpr1pG5wZIknUivrD1f/XygeU2wvw/SBBt2k0sKPqBAUSDrAXa4+bqwG+rbSSrwC64X5AtoSsyhsRa2pKivxxM+MsE1l2ui98IDwVlgaflWK/VRo0pi3RQQKq51F5fIQut51Nwhcx74+0c/5jrAEVYotSHo2jUwhWssw5CSbb/JTzT02eKVq1n1t1KymWzLENLrvuCUD6vrEcYZoisqstcLrrzetVjGotVCYdaOGso33Xnym1GGuE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings':

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 20.731626715s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

### Human-in-the-loop
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
 - High-stakes operations requiring human approval (e.g. database writes, financial transactions).
 - Compliance workflows where human oversight is mandatory.
 - Long-running conversations where human feedback guides the agent.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [13]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"] 
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [14]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [15]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='215ac810-7ec3-4f69-a065-f89e9b7b9e41'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_439147': 'EmAKXgFpFH0TIS6zXhXx67nqGQCx/d18P8d+STa/NBV2gQdD2yQ4vV5TkKBY634rcoUoV7ZGPy2zgX6u+Sjsf76Eg+RATsTT413Bo0ncvYq+heB2/0TMP5bQp4q+MABsVuY='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5df-9e3e-7df1-8361-ed4c9505682a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'call_439147', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144,

### If human wants to approve:

In [16]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'Email has been successfully sent to john@test.com with the subject "Hello".', 'extras': {'signature': 'EmAKXgFpFH0Tk37TONBdz37nweRkSisaaNaCeYJUYLtRps4DjFe1eNvKYQ1S27SygPivTG+pg+ZSFnrk4zihkHha4q+XZkp94nQ4j/MZCxktlRytjEXp5iQ2UDJbg6FOmAg='}}]


In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='215ac810-7ec3-4f69-a065-f89e9b7b9e41'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_439147': 'EmAKXgFpFH0TIS6zXhXx67nqGQCx/d18P8d+STa/NBV2gQdD2yQ4vV5TkKBY634rcoUoV7ZGPy2zgX6u+Sjsf76Eg+RATsTT413Bo0ncvYq+heB2/0TMP5bQp4q+MABsVuY='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5df-9e3e-7df1-8361-ed4c9505682a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'call_439147', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144,

### If human wants to reject

In [18]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"] 
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [20]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [21]:
from langgraph.types import Command
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email sending action was cancelled because the user rejected the tool call. Let me know if you need help with anything else!', 'extras': {'signature': 'EmAKXgFpFH0Tsob378ZjL9pNwYXTHgbnzTBJBfuOFsRFQIy8mEdrD56L5IECpdf06cn/+1hGWHrzpXhTXQWRyLomz0sRK9Q2vCqpnpXvedal3P7q2F+TRH6RJKpBHc1QQvQ='}}]


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='55a0ea68-1ea1-472a-b887-ce4721ed3ca3'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_569531': 'EmAKXgFpFH0T1sxFqvez2LkGHg8ciYvBUzNnJflX79nzGvwmKMaUDgma+vchv0KR6emhv3NZ75Zo0mf0d/ixnFYuD6uUcK3dGPRZkKliwdD/Ga7aUTnde+YOhQMYRbJNMDs='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5e5-3fbd-7231-8737-9d989b04c052-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_569531', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144,

### If human wants to edit:


In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [7]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"] 
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [8]:
config = {"configurable": {"thread_id": "test-edit"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [9]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d03294a0-3498-4c44-8950-929cee8a73e7'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "wrong@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_468963': 'EnMKcQFpFH0T8tj7pb+IZlm1G3ZuOPmi5rbylejv3eiiiLCOv0uo0qoeyKdOqVSvb4qVoKlZnsrPxPdmndsDSGqgoGbwMrVYS6rXtRljknayNYAYSMPaR8ksSeVPKu7tH3+biNmxqeRR9eCGXujIbdGQLBAI'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5eb-e156-79b0-b54b-542a8cdefe78-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'wrong@test.com', 'subject': 'Hello'}, 'id': 'call_468963', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [11]:
from langgraph.types import Command

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': "The email has been sent. Please note that a human reviewer intervened and updated the recipient to 'correct@email.com' and the subject to 'Corrected Subject' before it was sent.", 'extras': {'signature': 'EnMKcQFpFH0TBV11pFfuPN5Wnv+N+ERy5MroP01qnpovK2D68kXd/AXwyy/dnsmx4Qdo0vlc/5rRwwQHSZFHnkS8vx0uHzpC4iHjAxlt5juDnnLGcm+mLMS/AegfVp0VV8WWj/SStf4h16G3Xu1MTfYwVtPz'}}]


In [12]:
result

{'messages': [HumanMessage(content="Send email to wrong@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d03294a0-3498-4c44-8950-929cee8a73e7'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "wrong@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_468963': 'EnMKcQFpFH0T8tj7pb+IZlm1G3ZuOPmi5rbylejv3eiiiLCOv0uo0qoeyKdOqVSvb4qVoKlZnsrPxPdmndsDSGqgoGbwMrVYS6rXtRljknayNYAYSMPaR8ksSeVPKu7tH3+biNmxqeRR9eCGXujIbdGQLBAI'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e5eb-e156-79b0-b54b-542a8cdefe78-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'wrong@test.com', 'subject': 'Hello'}, 'id': 'call_468963', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met